In [4]:
import vitaldb
import pandas as pd
import os

output_dir = "../patient_raw_data"
os.makedirs(output_dir, exist_ok=True)

track_names = [
    "SNUADC/ECG_II",         # Downsampled to 1 Hz
    "SNUADC/PLETH",          # Downsampled to 1 Hz
    "Solar8000/HR",          # Native ~0.5 - 1 Hz
    "Solar8000/ART_SBP",     # Native ~0.5 - 1 Hz
    "Solar8000/ART_DBP",     # Native ~0.5 - 1 Hz
    "Solar8000/ART_MBP",     # Native ~0.5 - 1 Hz
    "Solar8000/PLETH_SPO2",  # Native ~0.5 - 1 Hz
    "Solar8000/RR_CO2",      # Native ~0.5 - 1 Hz
    "Solar8000/ETCO2",       # Native ~0.5 - 1 Hz
    "Primus/FIO2",           # Native ~1 Hz
    "Solar8000/BT"           # Native ~0.5 - 1 Hz
]

In [5]:
print("Fetching case IDs")
caseids = vitaldb.find_cases(['Solar8000/ART_MBP', 'Solar8000/HR'])[:100]
print(f"Found {len(caseids)} patients. Starting 1 Hz extraction...")

# Extract and align to 1Hz
for i, caseid in enumerate(caseids):
    try:
        vf = vitaldb.VitalFile(caseid, track_names)
        df = vf.to_pandas(track_names, interval=1)
        df.insert(0, 'Time_sec', range(0, len(df)))
        filename = os.path.join(output_dir, f"patient_{caseid}_1hz.csv")
        df.to_csv(filename, index=False)
        
        print(f"[{i+1}/100] Saved Patient {caseid} | Rows: {len(df)}")
        
    except Exception as e:
        print(f"Failed to process Patient {caseid}. Error: {e}")

print("Extraction complete!")

Fetching case IDs
Found 100 patients. Starting 1 Hz extraction...


C:\Users\Raj Jaiswal\AppData\Local\Temp\ipykernel_8036\1486908676.py:2: DeprecationWarning: find_cases() relies on the per-track 'trks' index, which is deprecated and may be removed in a future release.
  caseids = vitaldb.find_cases(['Solar8000/ART_MBP', 'Solar8000/HR'])[:100]


[1/100] Saved Patient 1 | Rows: 11543
[2/100] Saved Patient 3 | Rows: 4395
[3/100] Saved Patient 4 | Rows: 20990
[4/100] Saved Patient 7 | Rows: 15770
[5/100] Saved Patient 10 | Rows: 20993
[6/100] Saved Patient 12 | Rows: 31203
[7/100] Saved Patient 13 | Rows: 10812
[8/100] Saved Patient 14 | Rows: 4116
[9/100] Saved Patient 16 | Rows: 12868
[10/100] Saved Patient 17 | Rows: 20370
[11/100] Saved Patient 19 | Rows: 27576
[12/100] Saved Patient 20 | Rows: 26476
[13/100] Saved Patient 22 | Rows: 14375
[14/100] Saved Patient 24 | Rows: 6492
[15/100] Saved Patient 25 | Rows: 14833
[16/100] Saved Patient 26 | Rows: 10711
[17/100] Saved Patient 27 | Rows: 17870
[18/100] Saved Patient 28 | Rows: 26903
[19/100] Saved Patient 29 | Rows: 21394
[20/100] Saved Patient 31 | Rows: 10416
[21/100] Saved Patient 32 | Rows: 3920
[22/100] Saved Patient 34 | Rows: 23774
[23/100] Saved Patient 38 | Rows: 12291
[24/100] Saved Patient 41 | Rows: 11831
[25/100] Saved Patient 43 | Rows: 14790
[26/100] Saved Pa

In [7]:
import pandas as pd
import numpy as np
import os
import glob

input_dir = "../patient_raw_data"
output_dir = "../patient_labeled_data"
os.makedirs(output_dir, exist_ok=True)

csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
print(f"Found {len(csv_files)} files. Starting data cleaning and feature engineering...")

bounds = {
    'Solar8000/HR': (20, 300),         # Heart Rate
    'Solar8000/ART_SBP': (30, 300),    # Systolic BP
    'Solar8000/ART_DBP': (10, 200),    # Diastolic BP
    'Solar8000/ART_MBP': (20, 250),    # Mean BP
    'Solar8000/PLETH_SPO2': (50, 100), # SpO2 (%)
    'Solar8000/RR_CO2': (0, 60),       # Respiratory Rate
    'Solar8000/ETCO2': (0, 100),       # End-Tidal CO2
    'Primus/FIO2': (20, 100),          # FiO2 (%)
    'Solar8000/BT': (25, 45)           # Body Temp (Celsius)
}

forecast_window = 600 
for i, file_path in enumerate(csv_files):
    try:
        filename = os.path.basename(file_path)
        df = pd.read_csv(file_path)
        
        for col, (min_val, max_val) in bounds.items():
            if col in df.columns:
                df[col] = np.where((df[col] >= min_val) & (df[col] <= max_val), df[col], np.nan)
        
        features_to_fill = [col for col in bounds.keys() if col in df.columns]
        df[features_to_fill] = df[features_to_fill].ffill(limit=60)
    
        hr_rolling = df['Solar8000/HR'].rolling(window=5, min_periods=1).mean()
        mbp_rolling = df['Solar8000/ART_MBP'].rolling(window=5, min_periods=1).mean()
        spo2_rolling = df['Solar8000/PLETH_SPO2'].rolling(window=5, min_periods=1).mean()
    
        df['Target_Tachycardia'] = (hr_rolling > 110).astype(int)
        df['Target_Hypotension'] = (mbp_rolling < 65).astype(int)
        df['Target_Hypoxia'] = (spo2_rolling < 90).astype(int)
        
        df['Future_Tachycardia'] = df['Target_Tachycardia'].rolling(window=forecast_window, min_periods=1).max().shift(-forecast_window)
        df['Future_Hypotension'] = df['Target_Hypotension'].rolling(window=forecast_window, min_periods=1).max().shift(-forecast_window)
        df['Future_Hypoxia'] = df['Target_Hypoxia'].rolling(window=forecast_window, min_periods=1).max().shift(-forecast_window)
        
        df = df.dropna(subset=['Future_Tachycardia', 'Future_Hypotension', 'Future_Hypoxia'])        
        output_path = os.path.join(output_dir, filename)
        df.to_csv(output_path, index=False)
        
        print(f"[{i+1}/{len(csv_files)}] Cleaned and labeled: {filename}")
        
    except Exception as e:
        print(f"Failed to process {file_path}. Error: {e}")

print("Data cleaning and target generation complete! Cleaned files are in the output folder.")

Found 100 files. Starting data cleaning and feature engineering...
[1/100] Cleaned and labeled: patient_101_1hz.csv
[2/100] Cleaned and labeled: patient_103_1hz.csv
[3/100] Cleaned and labeled: patient_104_1hz.csv
[4/100] Cleaned and labeled: patient_105_1hz.csv
[5/100] Cleaned and labeled: patient_108_1hz.csv
[6/100] Cleaned and labeled: patient_10_1hz.csv
[7/100] Cleaned and labeled: patient_110_1hz.csv
[8/100] Cleaned and labeled: patient_111_1hz.csv
[9/100] Cleaned and labeled: patient_112_1hz.csv
[10/100] Cleaned and labeled: patient_113_1hz.csv
[11/100] Cleaned and labeled: patient_114_1hz.csv
[12/100] Cleaned and labeled: patient_115_1hz.csv
[13/100] Cleaned and labeled: patient_116_1hz.csv
[14/100] Cleaned and labeled: patient_117_1hz.csv
[15/100] Cleaned and labeled: patient_118_1hz.csv
[16/100] Cleaned and labeled: patient_119_1hz.csv
[17/100] Cleaned and labeled: patient_124_1hz.csv
[18/100] Cleaned and labeled: patient_125_1hz.csv
[19/100] Cleaned and labeled: patient_126_1